# Region + Sampler showcase

Interactive preview of `pybasin`'s `Region` + `Sampler` API for generating initial
conditions: pick a region, pick a sampler, and see the resulting initial conditions in 2D
or 3D, including the built-in incompatible combinations failing explicitly.

In [1]:
import itertools
from typing import cast

import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import display

from pybasin.regions import BallRegion, BoxRegion, SphereSurfaceRegion
from pybasin.samplers import (
    CartesianProductSampler,
    EvenlySpacedSampler,
    LatinHypercubeSampler,
    SobolSampler,
    TruncatedGaussianSampler,
    UniformRandomSampler,
    UnsupportedSamplerConfigurationError,
    UnsupportedSamplerRegionError,
)

torch.manual_seed(0)

## Compatibility matrix, exercised directly

Confirm every documented combination actually behaves as documented: supported pairs return points, unsupported pairs raise explicitly.

In [2]:
COMPATIBILITY = {
    "Box": ["UniformRandom", "EvenlySpaced", "TruncatedGaussian", "LatinHypercube", "Sobol"],
    "Ball": ["UniformRandom", "EvenlySpaced", "TruncatedGaussian"],
    "SphereSurface": ["UniformRandom", "EvenlySpaced"],
}

ReferenceRegion = BoxRegion | BallRegion | SphereSurfaceRegion


def available_samplers(region_type: str, dim: int) -> list[str]:
    """Compatible sampler names for (region_type, dim); EvenlySpaced on SphereSurface needs 3D."""
    names = list(COMPATIBILITY[region_type])
    if region_type == "SphereSurface" and dim != 3 and "EvenlySpaced" in names:
        names.remove("EvenlySpaced")
    return names


def reference_region(region_type: str, dim: int) -> ReferenceRegion:
    if region_type == "Box":
        return BoxRegion(torch.full((dim,), -2.0), torch.full((dim,), 2.0))
    if region_type == "Ball":
        return BallRegion(torch.zeros(dim), radius=2.0)
    return SphereSurfaceRegion(torch.zeros(dim), radius=2.0)


def try_sample(region_type: str, sampler_name: str, dim: int, n: int = 256) -> str:
    region = reference_region(region_type, dim)
    try:
        if sampler_name == "UniformRandom":
            points = UniformRandomSampler(region).sample(n)
        elif sampler_name == "EvenlySpaced":
            points = EvenlySpacedSampler(region).sample(n)
        elif sampler_name == "TruncatedGaussian":
            if isinstance(region, (BallRegion, SphereSurfaceRegion)):
                mean = region.center
            else:
                mean = (region.min + region.max) / 2
            std = 0.3 * torch.ones(dim)
            points = TruncatedGaussianSampler(region, mean=mean, std=std).sample(n)
        elif sampler_name == "LatinHypercube":
            points = LatinHypercubeSampler(region).sample(n)
        elif sampler_name == "Sobol":
            points = SobolSampler(region, seed=0).sample(n)
        else:
            raise ValueError(sampler_name)
    except (UnsupportedSamplerRegionError, UnsupportedSamplerConfigurationError) as exc:
        return f"raises {type(exc).__name__}"
    assert bool(region.contains(points).all()), "generated points violated region.contains"
    return f"ok, shape={tuple(points.shape)}"


ALL_SAMPLERS = ["UniformRandom", "EvenlySpaced", "TruncatedGaussian", "LatinHypercube", "Sobol"]
for dim in (2, 3):
    print(f"--- state_dim = {dim} ---")
    for region_type in ("Box", "Ball", "SphereSurface"):
        for sampler_name in ALL_SAMPLERS:
            expected_supported = sampler_name in available_samplers(region_type, dim)
            outcome = try_sample(region_type, sampler_name, dim)
            actually_supported = outcome.startswith("ok")
            assert actually_supported == expected_supported, (
                f"compatibility mismatch: {region_type}/{sampler_name}/dim={dim} -> {outcome}"
            )
            print(f"{region_type:14s} x {sampler_name:17s} -> {outcome}")

--- state_dim = 2 ---
Box            x UniformRandom     -> ok, shape=(256, 2)
Box            x EvenlySpaced      -> ok, shape=(256, 2)
Box            x TruncatedGaussian -> ok, shape=(256, 2)
Box            x LatinHypercube    -> ok, shape=(256, 2)
Box            x Sobol             -> ok, shape=(256, 2)
Ball           x UniformRandom     -> ok, shape=(256, 2)
Ball           x EvenlySpaced      -> ok, shape=(172, 2)
Ball           x TruncatedGaussian -> ok, shape=(256, 2)
Ball           x LatinHypercube    -> raises UnsupportedSamplerRegionError
Ball           x Sobol             -> raises UnsupportedSamplerRegionError
SphereSurface  x UniformRandom     -> ok, shape=(256, 2)
SphereSurface  x EvenlySpaced      -> raises UnsupportedSamplerConfigurationError
SphereSurface  x TruncatedGaussian -> raises UnsupportedSamplerRegionError
SphereSurface  x LatinHypercube    -> raises UnsupportedSamplerRegionError
SphereSurface  x Sobol             -> raises UnsupportedSamplerRegionError
--- stat

## Interactive explorer

Pick a state-space dimension (2D or 3D), a region, and a sampler. The sampler dropdown
narrows itself to the combinations the compatibility matrix above actually supports for
the current region + dimension (e.g. `EvenlySpaced` on a `SphereSurface` only appears in
3D, matching the Fibonacci-sphere restriction).

In [3]:
def box_edges_trace(
    min_bound: torch.Tensor, max_bound: torch.Tensor
) -> tuple[list[float | None], list[float | None], list[float | None]]:
    lo, hi = min_bound.tolist(), max_bound.tolist()
    bits = list(itertools.product([0, 1], repeat=3))
    corners = [
        (hi[0] if a else lo[0], hi[1] if b else lo[1], hi[2] if c else lo[2]) for a, b, c in bits
    ]
    edges = [
        (i, j)
        for i in range(8)
        for j in range(i + 1, 8)
        if sum(p != q for p, q in zip(bits[i], bits[j], strict=True)) == 1
    ]
    xs: list[float | None] = []
    ys: list[float | None] = []
    zs: list[float | None] = []
    for i, j in edges:
        xs += [corners[i][0], corners[j][0], None]
        ys += [corners[i][1], corners[j][1], None]
        zs += [corners[i][2], corners[j][2], None]
    return xs, ys, zs


def sphere_mesh(
    center: torch.Tensor, radius: float, resolution: int = 40
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    u = np.linspace(0, 2 * np.pi, resolution)
    v = np.linspace(0, np.pi, resolution // 2)
    c = center.tolist()
    x = radius * np.outer(np.cos(u), np.sin(v)) + c[0]
    y = radius * np.outer(np.sin(u), np.sin(v)) + c[1]
    z = radius * np.outer(np.ones_like(u), np.cos(v)) + c[2]
    return x, y, z


def make_figure(points: torch.Tensor, region: ReferenceRegion, dim: int, title: str) -> go.Figure:
    pts = points.detach().cpu().numpy()
    fig = go.Figure()
    if dim == 2:
        fig.add_trace(
            go.Scattergl(
                x=pts[:, 0],
                y=pts[:, 1],
                mode="markers",
                marker={"size": 4, "opacity": 0.55, "color": "#3b82f6"},
                name="samples",
            )
        )
        if isinstance(region, BoxRegion):
            x0, y0 = region.min.tolist()
            x1, y1 = region.max.tolist()
            fig.add_shape(
                type="rect",
                x0=x0,
                y0=y0,
                x1=x1,
                y1=y1,
                line={"color": "#ef4444", "dash": "dot", "width": 2},
            )
        else:
            cx, cy = region.center.tolist()
            theta = np.linspace(0, 2 * np.pi, 200)
            fig.add_trace(
                go.Scatter(
                    x=cx + region.radius * np.cos(theta),
                    y=cy + region.radius * np.sin(theta),
                    mode="lines",
                    line={"color": "#ef4444", "dash": "dot"},
                    name="boundary",
                )
            )
        fig.update_layout(
            width=560,
            height=560,
            title=title,
            xaxis={"scaleanchor": "y", "title": "x0"},
            yaxis={"title": "x1"},
        )
    else:
        fig.add_trace(
            go.Scatter3d(
                x=pts[:, 0],
                y=pts[:, 1],
                z=pts[:, 2],
                mode="markers",
                marker={"size": 2.5, "opacity": 0.55, "color": "#3b82f6"},
                name="samples",
            )
        )
        if isinstance(region, BoxRegion):
            xs, ys, zs = box_edges_trace(region.min, region.max)
            fig.add_trace(
                go.Scatter3d(
                    x=xs,
                    y=ys,
                    z=zs,
                    mode="lines",
                    line={"color": "#ef4444", "width": 3},
                    showlegend=False,
                )
            )
        else:
            x, y, z = sphere_mesh(region.center, region.radius)
            fig.add_trace(
                go.Surface(
                    x=x,
                    y=y,
                    z=z,
                    opacity=0.15,
                    showscale=False,
                    colorscale=[[0, "#ef4444"], [1, "#ef4444"]],
                )
            )
        fig.update_layout(width=650, height=600, title=title, scene={"aspectmode": "cube"})
    return fig

In [ ]:
dim_widget = widgets.ToggleButtons(options=[("2D", 2), ("3D", 3)], value=2, description="dim:")
region_widget = widgets.Dropdown(
    options=["Box", "Ball", "SphereSurface"], value="Box", description="region:"
)
sampler_widget = widgets.Dropdown(options=available_samplers("Box", 2), description="sampler:")
seed_widget = widgets.IntText(value=0, description="seed:")
n_widget = widgets.IntSlider(value=2000, min=100, max=8000, step=100, description="n:")

x_range_widget = widgets.FloatRangeSlider(
    value=[-2.0, 2.0], min=-5.0, max=5.0, step=0.1, description="x range:"
)
y_range_widget = widgets.FloatRangeSlider(
    value=[-2.0, 2.0], min=-5.0, max=5.0, step=0.1, description="y range:"
)
z_range_widget = widgets.FloatRangeSlider(
    value=[-2.0, 2.0], min=-5.0, max=5.0, step=0.1, description="z range:"
)
radius_widget = widgets.FloatSlider(value=2.0, min=0.1, max=5.0, step=0.1, description="radius:")

std_frac_widget = widgets.FloatSlider(
    value=0.3, min=0.05, max=1.0, step=0.05, description="std frac (TruncatedGaussian):"
)
scramble_widget = widgets.Checkbox(value=True, description="scramble (Sobol)")


def _sync_sampler_options(*_args: object) -> None:
    region_type = cast(str, region_widget.value)
    dim = cast(int, dim_widget.value)
    options = available_samplers(region_type, dim)
    previous = sampler_widget.value
    sampler_widget.options = options
    sampler_widget.value = previous if previous in options else options[0]


def _sync_dim_dependent_widgets(*_args: object) -> None:
    z_range_widget.disabled = dim_widget.value != 3


region_widget.observe(_sync_sampler_options, names="value")
dim_widget.observe(_sync_sampler_options, names="value")
dim_widget.observe(_sync_dim_dependent_widgets, names="value")
_sync_dim_dependent_widgets()


def build_region(region_type: str, dim: int) -> ReferenceRegion:
    if region_type == "Box":
        ranges = [x_range_widget.value, y_range_widget.value, z_range_widget.value][:dim]
        lo = torch.tensor([r[0] for r in ranges])
        hi = torch.tensor([r[1] for r in ranges])
        return BoxRegion(lo, hi)
    center = torch.zeros(dim)
    if region_type == "Ball":
        return BallRegion(center, radius=radius_widget.value)
    return SphereSurfaceRegion(center, radius=radius_widget.value)


def build_points(
    region: ReferenceRegion, sampler_name: str, dim: int, n: int, seed: int
) -> torch.Tensor:
    if sampler_name == "UniformRandom":
        return UniformRandomSampler(region).sample(n, seed=seed)
    if sampler_name == "EvenlySpaced":
        # n is approximate for Box/Ball (rounded up to the nearest even grid, then
        # filtered for Ball) and exact for SphereSurface -- see EvenlySpacedSampler's
        # docstring. seed doesn't apply: this sampler is deterministic.
        return EvenlySpacedSampler(region).sample(n)
    if sampler_name == "TruncatedGaussian":
        if isinstance(region, BoxRegion):
            mean = (region.min + region.max) / 2
            std = std_frac_widget.value * (region.max - region.min) / 2
        else:
            mean = region.center
            std = std_frac_widget.value * region.radius * torch.ones(dim)
        return TruncatedGaussianSampler(region, mean=mean, std=std).sample(n, seed=seed)
    if sampler_name == "LatinHypercube":
        return LatinHypercubeSampler(region).sample(n, seed=seed)
    if sampler_name == "Sobol":
        return SobolSampler(region, scramble=scramble_widget.value, seed=seed).sample(n)
    raise ValueError(sampler_name)


def render(
    dim: int,
    region_type: str,
    sampler_name: str,
    seed: int,
    n: int,
    x_range: tuple[float, float],
    y_range: tuple[float, float],
    z_range: tuple[float, float],
    radius: float,
    std_frac: float,
    scramble: bool,
) -> None:
    region = build_region(region_type, dim)
    try:
        points = build_points(region, sampler_name, dim, n, seed)
    except (
        UnsupportedSamplerRegionError,
        UnsupportedSamplerConfigurationError,
        RuntimeError,
        ValueError,
    ) as exc:
        print(f"{type(exc).__name__}: {exc}")
        return
    title = f"{region_type} x {sampler_name}  (requested n={n}, plotted n={points.shape[0]})"
    fig = make_figure(points, region, dim, title)
    fig.show()


controls = widgets.VBox([
    widgets.HBox([dim_widget, region_widget, sampler_widget]),
    widgets.HBox([seed_widget, n_widget]),
    widgets.HBox([x_range_widget, y_range_widget, z_range_widget]),
    widgets.HBox([radius_widget]),
    widgets.HBox([std_frac_widget, scramble_widget]),
])

out = widgets.interactive_output(
    render,
    {
        "dim": dim_widget,
        "region_type": region_widget,
        "sampler_name": sampler_widget,
        "seed": seed_widget,
        "n": n_widget,
        "x_range": x_range_widget,
        "y_range": y_range_widget,
        "z_range": z_range_widget,
        "radius": radius_widget,
        "std_frac": std_frac_widget,
        "scramble": scramble_widget,
    },
)

display(controls, out)

## Notes

- `EvenlySpaced` uses `n` like every other sampler; for `Box`/`Ball` it distributes `n`
  across active dimensions (`ceil(n ** (1 / n_active))` per axis), so the actual count is
  *approximately* `n` (see the title's "requested n" vs. "plotted n"), and for
  `SphereSurface` it is the exact Fibonacci-sphere point count. It ignores `seed`, since it
  is fully deterministic.
- Switching `region` to `SphereSurface` while in 2D removes `EvenlySpaced` from the sampler
  list, because Fibonacci-sphere spacing is only implemented for `state_dim == 3`.
- Switching `region` to `Ball` or `SphereSurface` removes `LatinHypercube` and `Sobol`:
  both are `BoxRegion`-only.
- `CsvSampler` isn't in this explorer. It doesn't take a `region` argument at all (its
  region is fitted from whatever CSV you load), so it doesn't fit the "pick any region,
  pick a compatible sampler" shape here. See its own section below.

## Exact grids: `CartesianProductSampler` and `n_points()`

`EvenlySpacedSampler(region).sample(n)` is *approximate*: `n` is distributed across active
dimensions and rounded up to the nearest even grid. For an exact grid built from explicit
per-axis coordinates instead, independent of the region's own bounds (`0:0.01:10` for one
dimension, `100:1:1000` for another), use `CartesianProductSampler`. Its point count is
entirely determined by `coordinates`, so `sample()` takes no `n` at all; there's no
meaningful "first n" of a Cartesian grid to slice. `n_points()` reports the resulting count
up front, the same "ask before you build it" role `CsvSampler.n_samples` plays for
CSV-backed data, for when you want to size `coordinates` before calling `sample`.

In [5]:
region = BoxRegion(torch.tensor([0.0, 100.0]), torch.tensor([10.0, 1000.0]))

# Explicit per-axis coordinates -- e.g. 0:0.01:10 for one dimension, 100:1:1000 for
# another (spelled out here as torch.arange since Python has no native range-with-float-step
# literal); independent of the region's bounds, filtered by region.contains() for safety.
grid = CartesianProductSampler(
    region, coordinates=[torch.arange(0.0, 10.01, 1.0), torch.arange(100.0, 1001.0, 100.0)]
)
print("n_points()     =", grid.n_points())
print("sample().shape =", tuple(grid.sample().shape))

# EvenlySpacedSampler's n is still the approximate, "just give me around this many" mode.
approx = EvenlySpacedSampler(region)
for n in (50, 200):
    print(
        f"n={n:<4d} -> n_points(n) =",
        approx.n_points(n),
        " sample(n).shape =",
        tuple(approx.sample(n).shape),
    )

n_points()     = 110
sample().shape = (110, 2)
n=50   -> n_points(n) = 64  sample(n).shape = (64, 2)
n=200  -> n_points(n) = 225  sample(n).shape = (225, 2)


## `CsvSampler`: region fitted from data, not supplied by the caller

`CsvSampler` loads initial conditions straight from a CSV file, which is useful for
reproducing exact results from MATLAB or other reference implementations. Its `region` is
a `BoxRegion` fitted to each coordinate column's own min/max, rather than something you
construct and pass in. Unlike every other sampler here, `sample(n)` doesn't randomize or
approximate: `n=None` returns every row in file order, and `n=<int>` returns the first
`n` rows.

In [6]:
import tempfile
from pathlib import Path

from pybasin.samplers import CsvSampler

csv_text = "x1,x2,label\n0.4,0.0,FP\n2.7,0.0,LC\n1.1,-0.3,FP\n-0.8,0.6,LC\n"
csv_path = Path(tempfile.mkdtemp()) / "pendulum_ground_truth.csv"
csv_path.write_text(csv_text)

csv_sampler = CsvSampler(csv_path, coordinate_columns=["x1", "x2"], label_column="label")
print("region        =", csv_sampler.region)
print("n_samples     =", csv_sampler.n_samples)
print("labels        =", csv_sampler.labels)
print("sample()      =\n", csv_sampler.sample())
print("sample(2)     =\n", csv_sampler.sample(2))

# CsvSampler.region is typed as the abstract Region protocol -- every sampler exposes the
# same member type there -- but is always fitted as a BoxRegion in practice.
csv_region = csv_sampler.region
assert isinstance(csv_region, BoxRegion)
fig = make_figure(csv_sampler.sample(), csv_region, 2, "CsvSampler (region fitted from the CSV)")
fig.show()

region        = BoxRegion(min=tensor([-0.8000, -0.3000], device='cuda:0'), max=tensor([2.7000, 0.6000], device='cuda:0'))
n_samples     = 4
labels        = ['FP' 'LC' 'FP' 'LC']
sample()      =
 tensor([[ 0.4000,  0.0000],
        [ 2.7000,  0.0000],
        [ 1.1000, -0.3000],
        [-0.8000,  0.6000]], device='cuda:0')
sample(2)     =
 tensor([[0.4000, 0.0000],
        [2.7000, 0.0000]], device='cuda:0')
